# Cancer-specific survival risk classification at 36 months in colorectal cancer

This notebook reproduces the complete machine learning analysis pipeline described in the manuscript.

The analysis workflow is organized into:
- cohort preparation
- feature engineering
- preprocessing
- model development
- calibration
- evaluation
- explainability
- sensitivity analyses

All analytical settings, random seeds, datasets, model configurations, and evaluation procedures are preserved from the original analysis.


In [ ]:
import os, re, sys, json, random, shutil, warnings, platform
from pathlib import Path
from datetime import datetime
import importlib.metadata as imd
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import joblib, optuna, shap
import xgboost as xgb, lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score, precision_score,
                             recall_score, f1_score, balanced_accuracy_score, confusion_matrix,
                             roc_curve, precision_recall_curve, brier_score_loss)
from lifelines import KaplanMeierFitter, CoxPHFitter, AalenJohansenFitter
from lifelines.utils import concordance_index
from lifelines.statistics import proportional_hazard_test
try:
    from sksurv.metrics import brier_score as sk_brier, cumulative_dynamic_auc as sk_cdauc
    from sksurv.util import Surv
    HAS_SKSURV = True
except ImportError:
    HAS_SKSURV = False
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---- config ----
SEED = 42
QUICK = os.getenv("QUICK_TEST_MODE", "0") == "1"   # default: full run
DATA_CANDIDATES = [os.getenv("CRC_DATA"), "NEW_CRC_2018-2020.txt"]
OUT = Path(os.getenv("CRC_OUT", "crc_output"))
N_FOLDS       = 2 if QUICK else 5
N_BOOT        = 20 if QUICK else 1000
OPTUNA_TRIALS = 2 if QUICK else 50
N_SHAP        = 100 if QUICK else 2000
MIN_SUBGROUP_N = 20 if QUICK else 100
QUICK_ROWS    = 15000
N_JOBS        = int(os.getenv("CRC_N_JOBS", "1"))   # 1 = identical numbers run to run; raise only to speed up
MISSING_FLAG  = 0.30
TARGET_COVERAGE = 0.90
THR_GRID = np.round(np.arange(0.05, 0.9501, 0.005), 3)
DCA_GRID = np.linspace(0.05, 0.60, 56)
TRAIN_YEARS, TEST_YEAR, HORIZON = [2018, 2019], 2020, 36
UNK = "Unknown/Not documented"

random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

# fresh output folders every run, so no stale files survive
for sub in ["tables", "tables/latex", "figures", "models"]:
    if sub in ("tables", "figures", "models") and (OUT / sub).exists():
        shutil.rmtree(OUT / sub)
    (OUT / sub).mkdir(parents=True, exist_ok=True)
TAB, TEX, FIG, MOD = OUT / "tables", OUT / "tables/latex", OUT / "figures", OUT / "models"

pkgs = ["numpy", "pandas", "scikit-learn", "xgboost", "lightgbm", "catboost", "shap", "optuna", "lifelines", "scikit-survival"]
VERSIONS = {"python": platform.python_version()}
for p in pkgs:
    try: VERSIONS[p] = imd.version(p)
    except Exception: VERSIONS[p] = "not installed"
print(VERSIONS)
print("QUICK mode:", QUICK)
if QUICK:
    print("WARNING: QUICK_TEST_MODE is ON (subsampled data, reduced tuning). Do not report these numbers.")
if not HAS_SKSURV:
    print("WARNING: scikit-survival is not installed; IPCW Brier and time-dependent AUC will be NaN.")

In [ ]:
R = {}          # every number quoted in the manuscript ends up here
CHECKS = []     # assertion log -> VERIFICATION.md
TABLES = {}

def check(name, ok, detail=""):
    CHECKS.append((name, bool(ok), str(detail)))
    print(("PASS  " if ok else "FAIL  ") + name, detail)
    assert ok, f"Check failed: {name} {detail}"

def save_table(df, name):
    df = df.copy()
    df.to_csv(TAB / f"{name}.csv", index=False)
    try:
        (TEX / f"{name}.tex").write_text(df.to_latex(index=False, float_format="%.3f", na_rep="--", escape=True))
    except Exception as e:
        print("latex skipped:", name, e)
    TABLES[name] = df

def save_fig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.show(); plt.close(fig)

def logit_clip(p, eps=1e-6):
    p = np.clip(np.asarray(p, float), eps, 1 - eps)
    return np.log(p / (1 - p))

def lr_unpenalized():
    return LogisticRegression(C=np.inf, solver="lbfgs", max_iter=2000)

def evaluate(y, p, thr=0.5):
    y = np.asarray(y).astype(int); p = np.asarray(p, float); pred = (p >= thr).astype(int)
    try: auc, ap = roc_auc_score(y, p), average_precision_score(y, p)
    except ValueError: auc = ap = np.nan
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"AUC-ROC": auc, "AUC-PR": ap, "Precision": precision_score(y, pred, zero_division=0),
            "Recall": recall_score(y, pred, zero_division=0), "F1": f1_score(y, pred, zero_division=0),
            "Accuracy": accuracy_score(y, pred), "Balanced accuracy": balanced_accuracy_score(y, pred),
            "Brier": brier_score_loss(y, p), "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}

def cal_slope_int(y, p):
    y = np.asarray(y).astype(int)
    if len(np.unique(y)) < 2: return np.nan, np.nan
    m = lr_unpenalized().fit(logit_clip(p).reshape(-1, 1), y)
    return float(m.coef_[0][0]), float(m.intercept_[0])

def ece(y, p, bins=10):
    y = np.asarray(y).astype(int); p = np.asarray(p, float)
    parts = np.array_split(np.argsort(p), bins)
    return float(sum(len(b) / len(p) * abs(y[b].mean() - p[b].mean()) for b in parts if len(b)))

def best_thr(y_death, p_death):
    f1s = [f1_score(y_death, (p_death >= t).astype(int), zero_division=0) for t in THR_GRID]
    return float(THR_GRID[int(np.argmax(f1s))])

# ---- bootstrap: same seed + same n -> same resampled indices, so comparisons are paired ----
def boot_iter(n, B, seed=SEED):
    rng = np.random.default_rng(seed)
    for _ in range(B): yield rng.integers(0, n, n)

def bootstrap(y, stat, B=None, seed=SEED):
    B = B or N_BOOT; y = np.asarray(y); out = []
    for idx in boot_iter(len(y), B, seed):
        if y[idx].min() == y[idx].max(): continue
        out.append(stat(idx))
    return np.asarray(out)

def ci(a):
    a = np.asarray(a); return float(a.mean()), float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))

def emp_p(d):
    d = np.asarray(d); return float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))

def fmt_ci(m, lo, hi, d=3): return f"{m:.{d}f} ({lo:.{d}f}-{hi:.{d}f})"

In [ ]:
YEAR, AGE, SEX, RACE, HISP = "Year of diagnosis", "Age recode with <1 year olds and 90+", "Sex", "Race recode (W, B, AI, API)", "Origin recode NHIA (Hispanic, Non-Hisp)"
PID, SEQ, INTL, NTUM = "Patient ID", "Sequence number", "Primary by international rules", "Total number of in situ/malignant tumors for patient"
SITE_RECODE, SITE_LABEL, HISTO, BEHAV = "Site recode ICD-O-3/WHO 2008", "Primary Site - labeled", "Histologic Type ICD-O-3", "Behavior recode for analysis"
GRADE, DIAG = "Derived Summary Grade 2018 (2018+)", "Diagnostic Confirmation"
SUMSTAGE, STAGEGRP = "Combined Summary Stage with Expanded Regional Codes (2004+)", "Derived EOD 2018 Stage Group Recode (2018+)"
T_COL, N_COL, M_COL = "Derived EOD 2018 T Recode (2018+)", "Derived EOD 2018 N Recode (2018+)", "Derived EOD 2018 M Recode (2018+)"
TSIZE, EOD_T, EOD_N, EOD_M = "Tumor Size Summary (2016+)", "EOD Primary Tumor Recode (2018+)", "EOD Regional Nodes Recode (2018+)", "EOD Mets Recode (2018+)"
NODES_EXAM, NODES_POS = "Regional nodes examined (1988+)", "Regional nodes positive (1988+)"
TUMOR_DEP, CEA, PNI = "Tumor Deposits Recode (2010+)", "CEA Pretreatment Interpretation Recode (2010+)", "Perineural Invasion Recode (2010+)"
SURV, VITAL, CSS_DEATH, COD = "Survival months", "Vital status recode (study cutoff used)", "SEER cause-specific death classification", "COD to site recode ICD-O-3 2023 Revision"
TARGET, TARGET_BEST, TARGET_WORST = "css_3yr_survival", "css_3yr_survival_best_case", "css_3yr_survival_worst_case"
SEQ_KEEP = {"One primary only", "1st of 2 or more primaries"}

MISSING_TOKENS = {"", " ", "Unknown", "unknown", "Blank(s)", "nan", "NaN", "None", "NA", "Not documented", "Not documented/assessed",
    "Not documented; Interpretation not assessed or unknown if assessed",
    "Not documented/assessed; No mention in path report; Pathologist cant determine",
    "Not documented/assessed; Indeterminate; No mention in path report; No resection", "Unable to calculate"}
# note: "88"/"99" are NOT global missing tokens (they are real values in size/node fields). They are handled per field below.

RAW_FILE = next((Path(p) for p in DATA_CANDIDATES if p and Path(p).exists()), None)
if RAW_FILE is None: raise FileNotFoundError(f"Raw SEER file not found. Tried: {DATA_CANDIDATES}. Set CRC_DATA.")
raw = pd.read_csv(RAW_FILE, dtype=str, keep_default_na=False, low_memory=False)
raw.columns = [c.strip() for c in raw.columns]
if QUICK: raw = raw.sample(QUICK_ROWS, random_state=SEED).reset_index(drop=True)
print(f"{RAW_FILE}: {raw.shape[0]:,} rows x {raw.shape[1]} cols; unique patients {raw[PID].nunique():,}")

parse_year = lambda s: pd.to_numeric(s.astype(str).str.extract(r"(\d{4})")[0], errors="coerce")
def overlap_ids(df):
    yr = parse_year(df[YEAR])
    return set(df.loc[yr.isin(TRAIN_YEARS), PID]) & set(df.loc[yr.eq(TEST_YEAR), PID])

steps = []
def step(name, before, after, note=""):
    steps.append({"Step": name, "Before N": int(before), "Excluded": int(before - after), "Remaining N": int(after), "Note": note})

d = raw.copy()
steps.append({"Step": "Raw SEER export", "Before N": len(d), "Excluded": 0, "Remaining N": len(d),
              "Note": "Malignant behaviour, known age and colon/rectum (excl. appendix) were applied in SEER*Stat"})
ov_raw = overlap_ids(d)

b = len(d)
ok = d[SITE_LABEL].str.contains(r"C18\.|C19\.|C20\.", na=False) & ~d[SITE_LABEL].str.contains(r"C18\.1", na=False)
d = d.loc[ok].reset_index(drop=True)
step("Strict C18-C20 sites, appendix (C18.1) excluded", b, len(d))
ov_site = overlap_ids(d)

b = len(d)
d = d.loc[d[SEQ].str.strip().isin(SEQ_KEEP)].reset_index(drop=True)
step("First primary only (Sequence number)", b, len(d), "kept: One primary only; 1st of 2 or more primaries")
ov_first = overlap_ids(d)

b = len(d)
if ov_first: d = d.loc[~d[PID].isin(ov_first)].reset_index(drop=True)
step("Drop residual train/test patient overlap", b, len(d), f"{len(ov_first)} overlapping patients after first-primary filter")

d["year_numeric"] = parse_year(d[YEAR])
d["survival_months"] = pd.to_numeric(d[SURV].str.strip(), errors="coerce")
d["death_event_crc"] = d[CSS_DEATH].str.strip().eq("Dead (attributable to this cancer dx)").astype(int)
d["unknown_cod_death"] = d[VITAL].str.strip().eq("Dead") & d[CSS_DEATH].str.strip().eq("Dead (missing/unknown COD)")

b = len(d)
lab = d.loc[~(d["survival_months"].isna() | d["unknown_cod_death"])].reset_index(drop=True)
step("Missing survival months or unknown cause of death", b, len(lab))

crc_early = lab["death_event_crc"].eq(1) & lab["survival_months"].lt(HORIZON)
obs36 = lab["survival_months"].ge(HORIZON)
lab[TARGET] = np.nan
lab.loc[crc_early, TARGET] = 0          # CRC death before 36 months
lab.loc[obs36, TARGET] = 1              # followed >= 36 months
lab[TARGET_BEST]  = np.where(crc_early, 0, 1)   # all censored -> survivor
lab[TARGET_WORST] = np.where(obs36, 1, 0)       # all censored -> non-survivor

b = len(lab)
n_def = int(lab[TARGET].notna().sum())
step("Undefined 3-year target (censored before 36 months)", b, n_def,
     "alive with <36 months follow-up, or non-CRC death (competing risk); kept for secondary analyses")

S1 = pd.DataFrame(steps)
S1 = pd.concat([S1, pd.DataFrame([
    {"Step": "Overlap check: raw extract", "Note": f"{len(ov_raw)} patients with records in both train years and 2020"},
    {"Step": "Overlap check: after site filter", "Note": f"{len(ov_site)} patients"},
    {"Step": "Overlap check: after first-primary filter", "Note": f"{len(ov_first)} patients"},
    {"Step": "Overlap check: final", "Note": "0 patients (asserted below)"}])], ignore_index=True)
save_table(S1, "TableS1_cohort_attrition")

yrs = []
for y_, g in lab.groupby("year_numeric"):
    yrs.append({"Year of diagnosis": int(y_), "N (before target step)": len(g), "N with defined target": int(g[TARGET].notna().sum()),
                "Excluded %": round(100 * (1 - g[TARGET].notna().mean()), 2)})
S2 = pd.DataFrame(yrs); save_table(S2, "TableS2_exclusion_share_by_year")
print(S1.to_string(index=False)); print(S2.to_string(index=False))
R.update({"raw_n": len(raw), "cohort_after_filters_n": len(d), "analytic_n": n_def, "overlap_raw": len(ov_raw), "overlap_after_first_primary": len(ov_first)})

In [ ]:
def cat(s, extra=()):
    s = s.astype(str).str.strip()
    return s.where(~(s.isin(MISSING_TOKENS) | s.isin(extra)), UNK)

def num(s): return pd.to_numeric(s.astype(str).str.strip(), errors="coerce")

def age_mid(v):
    s = str(v).strip()
    if not s or s in MISSING_TOKENS: return np.nan
    if "90+" in s: return 90.0
    if "<1" in s: return 0.0
    n = re.findall(r"\d+", s)
    return (float(n[0]) + float(n[1])) / 2 if len(n) >= 2 else (float(n[0]) if n else np.nan)

def site_group(v):
    s = str(v).strip()
    return s.split("-", 1)[1].strip() if "-" in s else (UNK if s in MISSING_TOKENS else s)

def simplify_stage(v):
    s = str(v).strip()
    if s in MISSING_TOKENS or s in {"88", "99", ""}: return "Unknown/unstaged"
    m = re.match(r"^([0-4])([ABC])?$", s)
    return f"Stage {dict(zip('01234', ['0','I','II','III','IV']))[m.group(1)]}" if m else s

def simplify_summary(v):
    s = str(v).strip().lower()
    for k, lab_ in [("localized", "Localized"), ("regional", "Regional"), ("distant", "Distant")]:
        if k in s: return lab_
    if "unknown" in s or "unstaged" in s or s in {"", "blank(s)"}: return "Unknown/unstaged"
    return str(v).strip()

# 2-tier grades are not forced into the 4-tier scale; rare levels are folded into Unknown
GRADE_MAP = {"Site-specific grade system category (1)": "Grade 1", "Site-specific grade system category (2)": "Grade 2",
             "Site-specific grade system category (3)": "Grade 3", "Site-specific grade system category (4)": "Grade 4",
             "Well differentiated": "Grade 1", "Moderately differentiated": "Grade 2", "Poorly differentiated": "Grade 3",
             "Undifferentiated and anaplastic": "Grade 4"}
def harmonize_grade(s, min_n=50):
    g = cat(s).map(GRADE_MAP).fillna("Unknown/indeterminate grade")
    small = g.value_counts()[lambda v: v < min_n].index
    return g.replace({c: "Unknown/indeterminate grade" for c in small})

def engineer(df):
    o = df.copy()
    o["age_numeric"] = o[AGE].apply(age_mid)
    o["sex_clean"], o["race_clean"], o["hispanic_origin_clean"] = cat(o[SEX]), cat(o[RACE]), cat(o[HISP])
    o["site_group"] = o[SITE_LABEL].apply(site_group)
    o["site_recode_clean"], o["histology_clean"] = cat(o[SITE_RECODE]), cat(o[HISTO])   # histology is nominal -> categorical only
    o["grade_clean"] = harmonize_grade(o[GRADE])
    o["diagnostic_confirmation_clean"] = cat(o[DIAG])
    o["summary_stage_clean"] = o[SUMSTAGE].apply(simplify_summary)
    o["stage_group_clean"] = o[STAGEGRP].apply(simplify_stage)
    # SEER "88" (not applicable) and "99" are treated as unknown, in T/N/M only
    for src, dst in [(T_COL, "t_stage_clean"), (N_COL, "n_stage_clean"), (M_COL, "m_stage_clean")]:
        o[dst] = cat(o[src], extra=("88", "99"))
    # EOD fields are nominal codes, not magnitudes -> categorical; 999 / 99 = unknown
    o["eod_primary_tumor_clean"] = cat(o[EOD_T], extra=("999",))
    o["eod_regional_nodes_clean"] = cat(o[EOD_N], extra=("999",))
    o["eod_mets_clean"] = cat(o[EOD_M], extra=("99",))
    # tumor size special codes
    sz = num(o[TSIZE])
    o["tumor_size_special"] = np.select(
        [sz.isna().to_numpy() | sz.eq(999).to_numpy(), sz.eq(0).to_numpy(), sz.eq(990).to_numpy(), sz.eq(998).to_numpy(), sz.between(1, 500).to_numpy()],
        ["Unknown/NA", "No mass/indeterminate", "Microscopic focus", "Polyposis/other", "Measured"], default="Implausible")
    o["tumor_size_mm"] = np.where(sz.between(1, 500), sz, np.where(sz.eq(990), 1.0, np.nan))
    # nodes: real counts are 0-90, everything else is a special code
    pos, exa = num(o[NODES_POS]), num(o[NODES_EXAM])
    o["regional_nodes_positive"] = pos.where(pos.between(0, 90))
    o["regional_nodes_examined"] = exa.where(exa.between(0, 90))
    o["nodal_assessment"] = np.select(
        [(pos.isna() | exa.isna()).to_numpy(), (pos.eq(98) | exa.eq(0)).to_numpy(), (pos.eq(99) | exa.eq(99)).to_numpy(),
         (pos.isin([95, 97]) | exa.isin([95, 96, 97, 98])).to_numpy()],
        ["Unknown", "No nodes examined", "Unknown", "Examined, count unspecified"], default="Examined, count known")
    e, p_ = o["regional_nodes_examined"], o["regional_nodes_positive"]
    o["node_ratio"] = (p_ / e).where((e >= 1) & (p_ <= e))
    o["cea_clean"], o["pni_clean"], o["tumor_deposits_clean"] = cat(o[CEA]), cat(o[PNI]), cat(o[TUMOR_DEP])
    return o

feat_full = engineer(lab)                                   # includes patients with undefined target (secondary analyses)
feat = feat_full.loc[feat_full[TARGET].notna()].reset_index(drop=True)
feat[TARGET] = feat[TARGET].astype(int)

PRIMARY_FEATURES = ["age_numeric", "sex_clean", "race_clean", "hispanic_origin_clean", "site_group", "site_recode_clean", "histology_clean",
    "grade_clean", "diagnostic_confirmation_clean", "summary_stage_clean", "stage_group_clean", "t_stage_clean", "n_stage_clean", "m_stage_clean",
    "tumor_size_mm", "tumor_size_special", "eod_primary_tumor_clean", "eod_regional_nodes_clean", "eod_mets_clean",
    "regional_nodes_examined", "regional_nodes_positive", "node_ratio", "nodal_assessment", "cea_clean", "pni_clean", "tumor_deposits_clean"]
NODE_PATHOLOGY_FEATURES = ["regional_nodes_examined", "regional_nodes_positive", "node_ratio", "nodal_assessment", "tumor_deposits_clean"]
PRETREATMENT_FEATURES = [c for c in PRIMARY_FEATURES if c not in NODE_PATHOLOGY_FEATURES]
POSTOP_FEATURES = list(PRIMARY_FEATURES)
# retained in the "pretreatment" set but reported with pathology/combined staging, so the label is only approximate
PRETREATMENT_CAVEATS = {
    "pni_clean": "perineural invasion comes from the pathology report", "grade_clean": "derived summary grade usually needs biopsy/resection pathology",
    "histology_clean": "final histology may reflect the resection specimen",
    "t_stage_clean / n_stage_clean / m_stage_clean / stage_group_clean / summary_stage_clean": "SEER derived EOD stage combines clinical and pathological information",
    "eod_primary_tumor_clean / eod_regional_nodes_clean / eod_mets_clean": "EOD recodes (incl. regional nodes) can include pathological findings",
    "tumor_size_mm / tumor_size_special": "tumor size summary can be taken from the pathology report", "cea_clean": "pretreatment lab value (genuinely pretreatment)"}

# EOD special-code report
eod_rep = pd.concat([feat[c].value_counts().rename_axis("code").reset_index(name="N").assign(variable=c)
                     for c in ["eod_primary_tumor_clean", "eod_regional_nodes_clean", "eod_mets_clean"]])
eod_rep.to_csv(TAB / "eod_code_report.csv", index=False)
print(eod_rep.groupby("variable").apply(lambda g: dict(zip(g.code, g.N))).to_string())
print(feat["nodal_assessment"].value_counts().to_string()); print(feat["tumor_size_special"].value_counts().to_string())

In [ ]:
train_m = feat["year_numeric"].isin(TRAIN_YEARS).to_numpy()
test_m  = feat["year_numeric"].eq(TEST_YEAR).to_numpy()
X_train, X_test = feat.loc[train_m, PRIMARY_FEATURES].reset_index(drop=True), feat.loc[test_m, PRIMARY_FEATURES].reset_index(drop=True)
y_train, y_test = feat.loc[train_m, TARGET].to_numpy(int), feat.loc[test_m, TARGET].to_numpy(int)
train_meta, test_meta = feat.loc[train_m].reset_index(drop=True), feat.loc[test_m].reset_index(drop=True)
yd_train, yd_test = 1 - y_train, 1 - y_test            # death within 36 months = positive class (headline)
NEG_POS = float((y_train == 0).sum() / (y_train == 1).sum())
print(f"train {len(X_train):,}  test {len(X_test):,}  death rate train {yd_train.mean():.3f} test {yd_test.mean():.3f}")

LEAKAGE = {SURV, VITAL, CSS_DEATH, COD, "survival_months", "death_event_crc", "unknown_cod_death", TARGET, TARGET_BEST, TARGET_WORST, YEAR, "year_numeric"}
check("nodes positive max <= 90", feat["regional_nodes_positive"].max() <= 90, feat["regional_nodes_positive"].max())
check("nodes examined max <= 90", feat["regional_nodes_examined"].max() <= 90, feat["regional_nodes_examined"].max())
check("tumor size max <= 500", feat["tumor_size_mm"].max() <= 500, feat["tumor_size_mm"].max())
check("no '88' in T/N/M", not any(feat[c].eq("88").any() for c in ["t_stage_clean", "n_stage_clean", "m_stage_clean"]))
ov_final = set(train_meta[PID]) & set(test_meta[PID])
check("train/test patient overlap == 0", len(ov_final) == 0, len(ov_final))
check("no leakage columns in features", not (set(PRIMARY_FEATURES) & LEAKAGE), "")
check("no constant features", all(feat[c].nunique(dropna=False) > 1 for c in PRIMARY_FEATURES))
check("pretreatment excludes nodal pathology", not (set(PRETREATMENT_FEATURES) & set(NODE_PATHOLOGY_FEATURES)))
print(f"features: primary {len(PRIMARY_FEATURES)}, pretreatment {len(PRETREATMENT_FEATURES)} (expected 26 / 21)")
R.update({"train_n": len(X_train), "test_n": len(X_test), "n_features_primary": len(PRIMARY_FEATURES), "n_features_pretreatment": len(PRETREATMENT_FEATURES),
          "death_rate_train": float(yd_train.mean()), "death_rate_test": float(yd_test.mean())})

In [ ]:
def smd_cont(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float); a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    sd = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return float((a.mean() - b.mean()) / sd) if sd > 0 else 0.0

def smd_prop(p1, p2):
    den = np.sqrt((p1 * (1 - p1) + p2 * (1 - p2)) / 2)
    return float((p1 - p2) / den) if den > 0 else 0.0

def baseline_table(cont, cats, rare_below=10):
    tr, te = train_meta, test_meta; n1, n2 = len(tr), len(te)
    h1, h2 = f"Train 2018-2019 (N={n1:,})", f"Test 2020 (N={n2:,})"
    rows = []
    for v in cont:
        a, b_ = tr[v].dropna(), te[v].dropna()
        rows.append({"Variable": v, "Level": "", h1: f"{a.mean():.1f} ± {a.std():.1f}", h2: f"{b_.mean():.1f} ± {b_.std():.1f}", "SMD": round(smd_cont(a, b_), 3)})
    for v in cats:
        tot = pd.concat([tr[v], te[v]]).value_counts()
        rare = [l for l in tot.index if tot[l] < rare_below]
        for l in sorted([l for l in tot.index if l not in rare], key=str):
            c1, c2 = (tr[v] == l).sum(), (te[v] == l).sum()
            rows.append({"Variable": v, "Level": l, h1: f"{c1:,} ({100*c1/n1:.1f}%)", h2: f"{c2:,} ({100*c2/n2:.1f}%)", "SMD": round(smd_prop(c1 / n1, c2 / n2), 3)})
        if rare:
            c1, c2 = tr[v].isin(rare).sum(), te[v].isin(rare).sum()
            rows.append({"Variable": v, "Level": f"Other/rare (<{rare_below} total)", h1: f"{c1:,} ({100*c1/n1:.1f}%)", h2: f"{c2:,} ({100*c2/n2:.1f}%)", "SMD": round(smd_prop(c1 / n1, c2 / n2), 3)})
    o1, o2 = tr[TARGET].mean(), te[TARGET].mean()
    rows.append({"Variable": "3-year CSS survivor (outcome)", "Level": "1", h1: f"{int(tr[TARGET].sum()):,} ({100*o1:.1f}%)", h2: f"{int(te[TARGET].sum()):,} ({100*o2:.1f}%)", "SMD": round(smd_prop(o1, o2), 3)})
    return pd.DataFrame(rows)

T2 = baseline_table(["age_numeric"], ["sex_clean", "race_clean", "hispanic_origin_clean", "stage_group_clean", "summary_stage_clean", "grade_clean", "nodal_assessment"])
S15 = baseline_table(["tumor_size_mm", "regional_nodes_examined", "regional_nodes_positive", "node_ratio"],
                     ["t_stage_clean", "n_stage_clean", "m_stage_clean", "site_group", "tumor_size_special", "eod_mets_clean"])
save_table(T2, "Table2_baseline_characteristics"); save_table(S15, "TableS15_baseline_detailed")
R["max_abs_smd_table2"] = float(T2["SMD"].abs().max())

SPECIAL_UNK = {"nodal_assessment": "Unknown", "tumor_size_special": "Unknown/NA"}
rows = []
for c in PRIMARY_FEATURES:
    s = feat[c]; numeric = pd.api.types.is_numeric_dtype(s)
    nm = int(s.isna().sum()) if numeric else int(s.eq(SPECIAL_UNK.get(c, UNK)).sum())
    rows.append({"Feature": c, "Type": "Numeric" if numeric else "Categorical", "N missing/unknown": nm, "N total": len(s),
                 "% missing": round(100 * nm / len(s), 2), f"> {int(MISSING_FLAG*100)}% missing": nm / len(s) > MISSING_FLAG})
S16 = pd.DataFrame(rows).sort_values("% missing", ascending=False).reset_index(drop=True)
save_table(S16, "TableS16_missingness_audit")
print(S16.head(10).to_string(index=False))
R["high_missing_features"] = S16.loc[S16.iloc[:, -1], "Feature"].tolist()

In [ ]:
MODEL_NAMES = ["Logistic Regression", "Random Forest", "XGBoost", "LightGBM", "CatBoost"]

def build_preprocessor(X):
    num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat_cols = [c for c in X.columns if c not in num_cols]
    try: ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=None if QUICK else 10)
    except TypeError: ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value=UNK)), ("oh", ohe)]), cat_cols)], remainder="drop")

def make_model(name, params=None):
    p = dict(params or {})
    if name == "Logistic Regression":
        d = dict(max_iter=3000, class_weight="balanced", solver="lbfgs", random_state=SEED)
        return LogisticRegression(**{**d, **p})
    if name == "Random Forest":
        d = dict(n_estimators=40 if QUICK else 250, min_samples_leaf=2, class_weight="balanced_subsample", n_jobs=N_JOBS, random_state=SEED, max_depth=8 if QUICK else None)
        return RandomForestClassifier(**{**d, **p})
    if name == "XGBoost":
        d = dict(n_estimators=40 if QUICK else 350, max_depth=3, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
                 tree_method="hist", n_jobs=N_JOBS, random_state=SEED, scale_pos_weight=NEG_POS)
        return xgb.XGBClassifier(**{**d, **p})
    if name == "LightGBM":
        d = dict(n_estimators=40 if QUICK else 450, learning_rate=0.04, num_leaves=31, subsample=0.9, colsample_bytree=0.9,
                 class_weight="balanced", n_jobs=N_JOBS, random_state=SEED, verbose=-1)
        return lgb.LGBMClassifier(**{**d, **p})
    if name == "CatBoost":
        d = dict(iterations=40 if QUICK else 450, depth=5, learning_rate=0.04, loss_function="Logloss", random_seed=SEED, thread_count=N_JOBS,
                 auto_class_weights="Balanced", verbose=False, allow_writing_files=False)
        return CatBoostClassifier(**{**d, **p})
    raise ValueError(name)

def pipe_for(X, name, params=None):
    return Pipeline([("preprocessor", build_preprocessor(X)), ("model", make_model(name, params))])

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def span(lo, hi): return (30, 60) if QUICK else (lo, hi)
SEARCH = {
    "Random Forest": lambda t: {"n_estimators": t.suggest_int("n_estimators", *span(150, 500)), "max_depth": t.suggest_int("max_depth", 4, 30),
                                "min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 10)},
    "XGBoost": lambda t: {"n_estimators": t.suggest_int("n_estimators", *span(150, 600)), "max_depth": t.suggest_int("max_depth", 2, 6),
                          "learning_rate": t.suggest_float("learning_rate", 0.01, 0.12, log=True), "subsample": t.suggest_float("subsample", 0.6, 1.0),
                          "colsample_bytree": t.suggest_float("colsample_bytree", 0.6, 1.0)},
    "LightGBM": lambda t: {"n_estimators": t.suggest_int("n_estimators", *span(150, 700)), "num_leaves": t.suggest_int("num_leaves", 15, 63),
                           "learning_rate": t.suggest_float("learning_rate", 0.01, 0.12, log=True), "subsample": t.suggest_float("subsample", 0.6, 1.0),
                           "colsample_bytree": t.suggest_float("colsample_bytree", 0.6, 1.0)},
    "CatBoost": lambda t: {"iterations": t.suggest_int("iterations", *span(150, 700)), "depth": t.suggest_int("depth", 3, 7),
                           "learning_rate": t.suggest_float("learning_rate", 0.01, 0.12, log=True)},
}
LR_GRID = [0.01, 0.1, 1.0] if QUICK else [0.001, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]

# NOTE: model selection and tuning both use the same CV folds on the training years (no nested CV).
# This can make the training-CV AUC slightly optimistic; it does not touch the locked 2020 test set.
BEST_PARAMS, OOF_RAW, CV_AUC = {}, {}, {}
for name in MODEL_NAMES:
    print("tuning", name, flush=True)
    best = {"auc": -1, "oof": None}
    def objective(trial, name=name, best=best):
        params = {"C": trial.suggest_categorical("C", LR_GRID)} if name == "Logistic Regression" else SEARCH[name](trial)
        oof = cross_val_predict(pipe_for(X_train, name, params), X_train, y_train, cv=cv, method="predict_proba", n_jobs=1)[:, 1]
        auc = roc_auc_score(y_train, oof)
        if auc > best["auc"]: best.update(auc=auc, oof=oof)
        return auc
    sampler = optuna.samplers.GridSampler({"C": LR_GRID}, seed=SEED) if name == "Logistic Regression" else optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=len(LR_GRID) if name == "Logistic Regression" else OPTUNA_TRIALS)
    BEST_PARAMS[name], OOF_RAW[name], CV_AUC[name] = study.best_params, best["oof"], best["auc"]
    print(f"  CV AUC {CV_AUC[name]:.4f}  {study.best_params}")

BEST = max(MODEL_NAMES, key=lambda m: CV_AUC[m])
print("Selected by tuned CV AUC:", BEST)
R.update({"best_model": BEST, "best_params": BEST_PARAMS[BEST], "cv_auc": {m: round(CV_AUC[m], 4) for m in MODEL_NAMES}})

S3 = pd.DataFrame([{"Model": m, "Tuned parameters": json.dumps(BEST_PARAMS[m]), **{k: v for k, v in evaluate(yd_train, 1 - OOF_RAW[m]).items()
                    if k in ["AUC-ROC", "AUC-PR", "Precision", "Recall", "F1", "Brier"]}} for m in MODEL_NAMES])
S3["Selected"] = S3["Model"].eq(BEST)
save_table(S3, "TableS3_cv_metrics_train_oof")
print(S3.drop(columns="Tuned parameters").to_string(index=False))

In [ ]:
# Platt calibrator is fit on each model's own training OOF predictions only
def fit_platt(Xtr, ytr, name, params, oof=None):
    pipe = pipe_for(Xtr, name, params)
    if oof is None: oof = cross_val_predict(pipe, Xtr, ytr, cv=cv, method="predict_proba", n_jobs=1)[:, 1]
    pipe.fit(Xtr, ytr)
    platt = lr_unpenalized().fit(logit_clip(oof).reshape(-1, 1), np.asarray(ytr).astype(int))
    return pipe, platt, platt.predict_proba(logit_clip(oof).reshape(-1, 1))[:, 1]

def score(pipe, platt, X):
    return platt.predict_proba(logit_clip(pipe.predict_proba(X)[:, 1]).reshape(-1, 1))[:, 1]

PIPES, PLATTS, P_RAW, P_SURV, OOF_SURV = {}, {}, {}, {}, {}
for m in MODEL_NAMES:
    PIPES[m], PLATTS[m], OOF_SURV[m] = fit_platt(X_train, y_train, m, BEST_PARAMS[m], oof=OOF_RAW[m])
    P_RAW[m] = PIPES[m].predict_proba(X_test)[:, 1]
    P_SURV[m] = PLATTS[m].predict_proba(logit_clip(P_RAW[m]).reshape(-1, 1))[:, 1]
    joblib.dump(PIPES[m], MOD / f"{m.replace(' ', '_').lower()}_pipeline.joblib"); joblib.dump(PLATTS[m], MOD / f"{m.replace(' ', '_').lower()}_platt.joblib")

BEST_PIPE, BEST_PLATT = PIPES[BEST], PLATTS[BEST]
pd_head = 1 - P_SURV[BEST]                         # P(death within 36 months), headline
pd_oof_head = 1 - OOF_SURV[BEST]
THR_F1 = best_thr(yd_train, pd_oof_head)           # chosen on training OOF only
print(f"death-class F1-optimal threshold (train OOF): {THR_F1}")

# one bootstrap for the headline: AUC-ROC, AUC-PR (death), ROC band
FPR_GRID = np.linspace(0, 1, 201)
def head_stat(idx):
    fpr, tpr, _ = roc_curve(yd_test[idx], pd_head[idx])
    return np.r_[roc_auc_score(yd_test[idx], pd_head[idx]), average_precision_score(yd_test[idx], pd_head[idx]), np.interp(FPR_GRID, fpr, tpr)]
hb = bootstrap(yd_test, head_stat)
auc_m, auc_lo, auc_hi = ci(hb[:, 0]); ap_m, ap_lo, ap_hi = ci(hb[:, 1])
tpr_lo, tpr_hi = np.percentile(hb[:, 2:], 2.5, axis=0), np.percentile(hb[:, 2:], 97.5, axis=0)

cal_s, cal_i = cal_slope_int(yd_test, pd_head)
rows = []
for thr, lab_ in [(0.50, "0.50 (default)"), (THR_F1, f"{THR_F1:.3f} (death-class F1-optimal, train OOF)")]:
    rows.append({"Threshold": lab_, **evaluate(yd_test, pd_head, thr)})
T4 = pd.DataFrame(rows)
T4["AUC-ROC 95% CI"] = fmt_ci(evaluate(yd_test, pd_head)["AUC-ROC"], auc_lo, auc_hi)
T4["AUC-PR 95% CI"] = fmt_ci(ap_m, ap_lo, ap_hi)
T4["Calibration slope"], T4["Calibration intercept"] = round(cal_s, 3), round(cal_i, 3)
T4["ECE"] = round(ece(yd_test, pd_head), 4)
save_table(T4, "Table4_headline_death_class_metrics")
print(T4.drop(columns=["TN", "FP", "FN", "TP"]).to_string(index=False))

R.update({"headline_auc_roc": float(roc_auc_score(yd_test, pd_head)), "headline_auc_roc_ci": [auc_lo, auc_hi],
          "headline_auc_pr_death": float(average_precision_score(yd_test, pd_head)), "headline_auc_pr_death_ci": [ap_lo, ap_hi],
          "headline_brier": float(brier_score_loss(yd_test, pd_head)), "f1_threshold_death": THR_F1,
          "headline_at_0.50": {k: float(v) for k, v in evaluate(yd_test, pd_head, 0.5).items()},
          "headline_at_f1_thr": {k: float(v) for k, v in evaluate(yd_test, pd_head, THR_F1).items()},
          "calibration_slope_headline": cal_s, "calibration_intercept_headline": cal_i, "ece_headline": ece(yd_test, pd_head)})

# model comparison (Table 3): paired bootstrap of AUC difference, best minus other
rows = []
for m in MODEL_NAMES:
    pdm = 1 - P_SURV[m]; ev = evaluate(yd_test, pdm)
    row = {"Model": m + (" (selected)" if m == BEST else ""), "CV AUC-ROC (train OOF)": round(CV_AUC[m], 4), "Test AUC-ROC": round(ev["AUC-ROC"], 4),
           "Test AUC-PR (death)": round(ev["AUC-PR"], 4), "Test Brier": round(ev["Brier"], 4)}
    if m == BEST: row.update({"AUC diff (best - model)": "ref", "95% CI": "", "Empirical p": ""})
    else:
        d_ = bootstrap(yd_test, lambda i, a=pdm: roc_auc_score(yd_test[i], pd_head[i]) - roc_auc_score(yd_test[i], a[i]))
        mm, lo, hi = ci(d_); row.update({"AUC diff (best - model)": round(mm, 4), "95% CI": f"{lo:.4f} to {hi:.4f}", "Empirical p": round(emp_p(d_), 4)})
    rows.append(row)
T3 = pd.DataFrame(rows); save_table(T3, "Table3_model_comparison"); print(T3.to_string(index=False))
R["model_comparison"] = T3.astype(str).to_dict("records")

In [ ]:
# raw vs isotonic vs Platt, all in death-class framing, all fit on training OOF only
iso = IsotonicRegression(out_of_bounds="clip").fit(OOF_RAW[BEST], y_train)
joblib.dump(iso, MOD / "best_isotonic.joblib")
cal_variants = {"Raw class-weighted": 1 - P_RAW[BEST], "Isotonic (train OOF)": 1 - iso.predict(P_RAW[BEST]), "Platt (headline)": pd_head}
rows = []
for k, p in cal_variants.items():
    s, i = cal_slope_int(yd_test, p)
    rows.append({"Probability": k, "Slope": round(s, 3), "Intercept": round(i, 3), "ECE": round(ece(yd_test, p), 4), "Brier": round(brier_score_loss(yd_test, p), 4),
                 "Mean predicted death risk": round(float(p.mean()), 4), "Observed death rate": round(float(yd_test.mean()), 4)})
S5 = pd.DataFrame(rows); save_table(S5, "TableS5_calibration_by_method"); print(S5.to_string(index=False))
gap = float(pd_head.mean() - yd_test.mean())
direction = ("over-estimates" if gap > 0 else "under-estimates") + f" overall death risk by {abs(gap)*100:.1f} percentage points"
slope_txt = "predictions slightly too extreme (slope < 1)" if cal_s < 1 else "predictions slightly too moderate (slope > 1)"
print(f"After Platt recalibration the model {direction}; calibration slope {cal_s:.3f}: {slope_txt}.")
R.update({"calibration_direction": direction, "calibration_slope_note": slope_txt, "calibration_table": S5.to_dict("records")})

fig, ax = plt.subplots(figsize=(6, 6))
for (k, p), c in zip(cal_variants.items(), ["#7f7f7f", "#ff7f0e", "#1f77b4"]):
    f, mp = calibration_curve(yd_test, p, n_bins=10, strategy="quantile"); ax.plot(mp, f, "o-", color=c, label=k, lw=2.2 if "Platt" in k else 1.3)
ax.plot([0, 1], [0, 1], "--", color="green", label="Perfect"); ax.set_xlabel("Mean predicted risk of death within 36 months"); ax.set_ylabel("Observed death fraction")
ax.legend(fontsize=9); save_fig(fig, "Fig_calibration")

# ROC with bootstrap band for the headline, other models plain
AUC_BY = {m: roc_auc_score(yd_test, 1 - P_SURV[m]) for m in MODEL_NAMES}
colors = dict(zip(MODEL_NAMES, plt.rcParams["axes.prop_cycle"].by_key()["color"]))
fig, ax = plt.subplots(figsize=(7, 7))
for m in sorted(MODEL_NAMES, key=lambda m: -AUC_BY[m]):
    if m == BEST: continue
    f, t, _ = roc_curve(yd_test, 1 - P_SURV[m]); ax.plot(f, t, color=colors[m], lw=1.3, label=f"{m} (AUC={AUC_BY[m]:.3f})")
ax.fill_between(FPR_GRID, tpr_lo, tpr_hi, color=colors[BEST], alpha=0.2)
f, t, _ = roc_curve(yd_test, pd_head); ax.plot(f, t, color=colors[BEST], lw=2.6, label=f"{BEST} (headline, AUC={AUC_BY[BEST]:.3f}; 95% CI band)")
ax.plot([0, 1], [0, 1], "--", color="grey", label="Chance"); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate (death within 36 months)")
ax.set_aspect("equal"); ax.grid(ls=":", alpha=0.35); ax.legend(fontsize=7, loc="lower right")
ins = ax.inset_axes([0.45, 0.30, 0.4, 0.4])
for m in MODEL_NAMES:
    f_, t_, _ = roc_curve(yd_test, 1 - P_SURV[m]); ins.plot(f_, t_, color=colors[m], lw=2 if m == BEST else 1)
ins.set_xlim(0, 0.3); ins.set_ylim(0.65, 0.95); ins.tick_params(labelsize=7); ax.indicate_inset_zoom(ins, edgecolor="grey")
save_fig(fig, "Fig_roc")

# PR curves (death class)
prev = float(yd_test.mean())
fig, ax = plt.subplots(figsize=(7, 6))
pr, rc, _ = precision_recall_curve(yd_test, pd_head)
ax.plot(rc, pr, lw=2, label=f"{BEST} (AUC-PR={R['headline_auc_pr_death']:.3f})")
for thr, mk, lb in [(0.5, "o", "threshold 0.50"), (THR_F1, "s", f"F1-optimal {THR_F1:.3f}")]:
    e = evaluate(yd_test, pd_head, thr); ax.scatter(e["Recall"], e["Precision"], marker=mk, s=70, zorder=5, label=lb)
ax.axhline(prev, ls="--", color="grey", label=f"No skill ({prev:.3f})"); ax.set_xlabel("Recall (death within 36 months)"); ax.set_ylabel("Precision")
ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02); ax.legend(fontsize=8, loc="lower left"); save_fig(fig, "Fig_pr_death")

fig, ax = plt.subplots(figsize=(7.5, 6))
for m in MODEL_NAMES:
    p_, r_, _ = precision_recall_curve(yd_test, 1 - P_SURV[m]); ax.plot(r_, p_, label=f"{m} (AUC-PR={average_precision_score(yd_test, 1 - P_SURV[m]):.3f})")
ax.axhline(prev, ls=":", color="black", label="No skill"); ax.set_xlabel("Recall (death within 36 months)"); ax.set_ylabel("Precision"); ax.legend(fontsize=8, loc="lower left")
save_fig(fig, "SF1_pr_all_models")

In [ ]:
TNM_ONLY = ["t_stage_clean", "n_stage_clean", "m_stage_clean", "stage_group_clean", "summary_stage_clean"]
TNM_CLIN = TNM_ONLY + ["age_numeric", "sex_clean", "site_group"]
base = {}
for key, fl in [("TNM only", TNM_ONLY), ("TNM + basic clinical", TNM_CLIN)]:
    pipe, platt, oof_p = fit_platt(X_train[fl], y_train, "Logistic Regression", {"C": BEST_PARAMS["Logistic Regression"]["C"]})
    base[key] = {"pipe": pipe, "platt": platt, "pd_test": 1 - score(pipe, platt, X_test[fl]), "pd_oof": 1 - oof_p}
    base[key]["thr"] = best_thr(yd_train, base[key]["pd_oof"])
pd_tnm, pd_clin = base["TNM only"]["pd_test"], base["TNM + basic clinical"]["pd_test"]

rows = []
for lab_, p_, thr_ in [("Model 1: TNM/stage only", pd_tnm, base["TNM only"]["thr"]), ("Model 2: TNM + age/sex/site", pd_clin, base["TNM + basic clinical"]["thr"]),
                       (f"Model 3: full model ({BEST})", pd_head, THR_F1)]:
    e5, ef = evaluate(yd_test, p_, 0.5), evaluate(yd_test, p_, thr_)
    rows.append({"Model": lab_, "AUC-ROC": e5["AUC-ROC"], "AUC-PR": e5["AUC-PR"], "Brier": e5["Brier"], "Precision@0.50": e5["Precision"], "Recall@0.50": e5["Recall"],
                 "F1@0.50": e5["F1"], "Threshold (F1-opt)": thr_, "Precision@F1thr": ef["Precision"], "Recall@F1thr": ef["Recall"], "F1@F1thr": ef["F1"]})
T5 = pd.DataFrame(rows)
dd = bootstrap(yd_test, lambda i: roc_auc_score(yd_test[i], pd_head[i]) - roc_auc_score(yd_test[i], pd_clin[i]))
dm, dlo, dhi = ci(dd)
T5["Delta AUC vs full (full - model)"] = ["", f"{dm:+.4f} ({dlo:+.4f} to {dhi:+.4f})", "ref"]
save_table(T5, "Table5_staging_baselines_vs_full"); print(T5.round(4).to_string(index=False))
R.update({"auc_tnm_only": T5.loc[0, "AUC-ROC"], "auc_tnm_clinical": T5.loc[1, "AUC-ROC"], "delta_auc_full_minus_tnm_clinical": [dm, dlo, dhi]})

# decision curves (event = death within 36 months)
def nb_model(y, p, pts):
    n = len(y); out = []
    for pt in pts:
        pr_ = p >= pt; out.append((pr_ & (y == 1)).sum() / n - (pr_ & (y == 0)).sum() / n * pt / (1 - pt))
    return np.array(out)
nb_all = lambda y, pts: y.mean() - (1 - y.mean()) * pts / (1 - pts)
dca = {"Full model": pd_head, "TNM + age/sex/site": pd_clin, "TNM/stage only": pd_tnm}
fig, ax = plt.subplots(figsize=(7.5, 6)); tabs = []
for k, p in dca.items():
    nb = nb_model(yd_test, p, DCA_GRID); ax.plot(DCA_GRID, nb, label=k); tabs.append(pd.DataFrame({"Threshold": DCA_GRID, "Net benefit": nb, "Model": k}))
ax.plot(DCA_GRID, nb_all(yd_test, DCA_GRID), "--", color="grey", label="Treat all"); ax.axhline(0, ls=":", color="k", label="Treat none")
ax.set_xlabel("Threshold probability (death within 36 months)"); ax.set_ylabel("Net benefit"); ax.legend(fontsize=8); save_fig(fig, "Fig_dca")
save_table(pd.concat(tabs), "Table_dca_net_benefit_curve")

pts = np.array([0.10, 0.20, 0.30, 0.40])
nbd = bootstrap(yd_test, lambda i: nb_model(yd_test[i], pd_head[i], pts) - nb_model(yd_test[i], pd_clin[i], pts))
dca_ci = pd.DataFrame({"Threshold": pts, "NB full": nb_model(yd_test, pd_head, pts), "NB TNM+clinical": nb_model(yd_test, pd_clin, pts),
                       "NB diff (full - TNM+clin)": nbd.mean(0), "95% CI low": np.percentile(nbd, 2.5, 0), "95% CI high": np.percentile(nbd, 97.5, 0)})
save_table(dca_ci, "Table_dca_netbenefit_difference_ci"); print(dca_ci.round(4).to_string(index=False))
R["dca_nb_difference"] = dca_ci.round(5).to_dict("records")

In [ ]:
stg = test_meta["stage_group_clean"].to_numpy()
r6, r7 = [], []
for s in sorted(pd.unique(stg)):
    m_ = stg == s; yy = yd_test[m_]
    if m_.sum() < MIN_SUBGROUP_N or len(np.unique(yy)) < 2: continue
    for lab_, p_, thr_ in [("Full model", pd_head, THR_F1), ("TNM + age/sex/site", pd_clin, base["TNM + basic clinical"]["thr"])]:
        e5, ef = evaluate(yy, p_[m_], 0.5), evaluate(yy, p_[m_], thr_)
        r6.append({"Stage": s, "Model": lab_, "N": int(m_.sum()), "Deaths": int(yy.sum()), "Few deaths (<30)": int(yy.sum()) < 30, "AUC-ROC": e5["AUC-ROC"], "AUC-PR": e5["AUC-PR"],
                   "Brier": e5["Brier"], "Precision@0.50": e5["Precision"], "Recall@0.50": e5["Recall"], "Precision@F1thr": ef["Precision"], "Recall@F1thr": ef["Recall"]})
    ds = bootstrap(yy, lambda i, m_=m_, yy=yy: roc_auc_score(yy[i], pd_head[m_][i]) - roc_auc_score(yy[i], pd_clin[m_][i]))
    a, lo, hi = ci(ds); r7.append({"Stage": s, "N": int(m_.sum()), "Deaths": int(yy.sum()), "Delta AUC (full - TNM+clin)": a, "95% CI low": lo, "95% CI high": hi, "CI excludes 0": (lo > 0) or (hi < 0)})
S6, S7 = pd.DataFrame(r6), pd.DataFrame(r7)
save_table(S6, "TableS6_stage_wise_performance"); save_table(S7, "TableS7_stage_wise_delta_auc"); print(S7.round(4).to_string(index=False))

# subgroups: race, sex, age band
sub = test_meta.copy(); sub["yd"], sub["pd"] = yd_test, pd_head
sub["age_band"] = pd.cut(sub["age_numeric"], [-0.1, 49, 64, 79, 200], labels=["<50", "50-64", "65-79", "80+"]).astype(str)
r8 = []
for v in ["race_clean", "sex_clean", "age_band"]:
    for l, g in sub.groupby(v):
        if len(g) < MIN_SUBGROUP_N: continue
        e = evaluate(g["yd"], g["pd"]); s_, i_ = cal_slope_int(g["yd"], g["pd"])
        r8.append({"Variable": v, "Level": l, "N": len(g), "Deaths": int(g["yd"].sum()), "AUC-ROC": e["AUC-ROC"], "Brier": e["Brier"], "Cal. slope": s_, "ECE": ece(g["yd"], g["pd"])})
S8 = pd.DataFrame(r8); save_table(S8, "TableS8_subgroup_performance")
spread_all = float(S8["AUC-ROC"].max() - S8["AUC-ROC"].min())
S8x = S8[~((S8["Variable"] == "race_clean") & (S8["Level"] == UNK))]
spread_x = float(S8x["AUC-ROC"].max() - S8x["AUC-ROC"].min())
print(S8.round(4).to_string(index=False)); print(f"AUC spread all subgroups {spread_all:.4f}; excluding Unknown race {spread_x:.4f}")
R.update({"subgroup_auc_spread_all": spread_all, "subgroup_auc_spread_excl_unknown_race": spread_x})

# pretreatment-only vs postoperative (same family and params)
pipe_pre, platt_pre, oof_pre = fit_platt(X_train[PRETREATMENT_FEATURES], y_train, BEST, BEST_PARAMS[BEST])
pd_pre = 1 - score(pipe_pre, platt_pre, X_test[PRETREATMENT_FEATURES]); thr_pre = best_thr(yd_train, 1 - oof_pre)
dpre = bootstrap(yd_test, lambda i: roc_auc_score(yd_test[i], pd_head[i]) - roc_auc_score(yd_test[i], pd_pre[i])); dpm, dpl, dph = ci(dpre)
rows = []
for lab_, p_, thr_, nf in [("Pretreatment-only (secondary)", pd_pre, thr_pre, len(PRETREATMENT_FEATURES)), ("Postoperative-inclusive (primary)", pd_head, THR_F1, len(POSTOP_FEATURES))]:
    rows.append({"Model": lab_, "N features": nf, **{f"{k}@0.50": v for k, v in evaluate(yd_test, p_).items() if k in ["AUC-ROC", "AUC-PR", "Brier", "Precision", "Recall", "F1"]},
                 "Threshold (F1-opt)": thr_, "F1@F1thr": evaluate(yd_test, p_, thr_)["F1"]})
S9 = pd.DataFrame(rows); S9["Delta AUC (post - pre)"] = ["", f"{dpm:+.4f} ({dpl:+.4f} to {dph:+.4f})"]
save_table(S9, "TableS9_pretreatment_vs_postoperative"); print(S9.round(4).to_string(index=False))
pd.DataFrame([{"Variable(s)": k, "Why it is not strictly pretreatment": v} for k, v in PRETREATMENT_CAVEATS.items()]).to_csv(TAB / "pretreatment_set_caveats.csv", index=False)
R.update({"auc_pretreatment": float(roc_auc_score(yd_test, pd_pre)), "delta_auc_post_minus_pre": [dpm, dpl, dph]})

In [ ]:
# cross-conformal: nonconformity scores from the 5-fold training OOF Platt probabilities
s_oof, s_test = OOF_SURV[BEST], P_SURV[BEST]
nc = np.where(y_train == 1, 1 - s_oof, s_oof)
q_level = min(np.ceil((len(nc) + 1) * TARGET_COVERAGE) / len(nc), 1.0)
qhat = float(np.quantile(nc, q_level, method="higher"))
in1, in0 = (1 - s_test) <= qhat, s_test <= qhat
size = in0.astype(int) + in1.astype(int)
covered = np.where(y_test == 1, in1, in0); abstain = size != 1
rows = [{"Group": "Overall", "N": len(y_test), "Empirical coverage": covered.mean(), "Abstain rate": abstain.mean()}]
for lab_, m_ in [("True survivors", y_test == 1), ("True deaths", y_test == 0)]:
    rows.append({"Group": lab_, "N": int(m_.sum()), "Empirical coverage": covered[m_].mean(), "Abstain rate": abstain[m_].mean()})
for s in sorted(pd.unique(stg)):
    m_ = stg == s
    if m_.sum() >= MIN_SUBGROUP_N: rows.append({"Group": f"Stage: {s}", "N": int(m_.sum()), "Empirical coverage": covered[m_].mean(), "Abstain rate": abstain[m_].mean()})
S10 = pd.DataFrame(rows); S10.insert(1, "Target coverage", TARGET_COVERAGE); S10["qhat"] = round(qhat, 4)
save_table(S10, "TableS10_conformal"); print(S10.round(4).to_string(index=False))
R.update({"conformal_coverage": float(covered.mean()), "conformal_abstain_rate": float(abstain.mean()), "conformal_qhat": qhat})

In [ ]:
LABELS = {"age_numeric": "Age", "sex_clean": "Sex", "race_clean": "Race", "hispanic_origin_clean": "Hispanic origin", "site_group": "Primary site",
    "site_recode_clean": "Primary site (SEER recode)", "histology_clean": "Histology", "grade_clean": "Grade", "diagnostic_confirmation_clean": "Diagnostic confirmation",
    "summary_stage_clean": "Summary stage", "stage_group_clean": "AJCC stage group", "t_stage_clean": "T stage", "n_stage_clean": "N stage", "m_stage_clean": "M stage",
    "tumor_size_mm": "Tumor size (mm)", "tumor_size_special": "Tumor size category", "eod_primary_tumor_clean": "EOD primary tumor", "eod_regional_nodes_clean": "EOD regional nodes",
    "eod_mets_clean": "EOD mets", "regional_nodes_examined": "Nodes examined", "regional_nodes_positive": "Nodes positive", "node_ratio": "Node ratio",
    "nodal_assessment": "Nodal assessment", "cea_clean": "CEA", "pni_clean": "Perineural invasion", "tumor_deposits_clean": "Tumor deposits"}
DOMAIN = {**{k: "Demographic" for k in ["age_numeric", "sex_clean", "race_clean", "hispanic_origin_clean"]},
    **{k: "Staging" for k in ["summary_stage_clean", "stage_group_clean", "t_stage_clean", "n_stage_clean", "m_stage_clean", "eod_primary_tumor_clean", "eod_regional_nodes_clean", "eod_mets_clean"]},
    **{k: "Pathology" for k in ["regional_nodes_examined", "regional_nodes_positive", "node_ratio", "nodal_assessment", "tumor_deposits_clean", "tumor_size_mm", "tumor_size_special", "pni_clean", "grade_clean", "histology_clean"]},
    "cea_clean": "Biomarker", "site_group": "Anatomic", "site_recode_clean": "Anatomic", "diagnostic_confirmation_clean": "Diagnostic"}
DCOL = {"Demographic": "#4477AA", "Staging": "#CC6677", "Pathology": "#228833", "Biomarker": "#CCBB44", "Anatomic": "#AA3377", "Diagnostic": "#66CCEE"}

pre, mdl = BEST_PIPE.named_steps["preprocessor"], BEST_PIPE.named_steps["model"]
enc_names = [re.sub(r"^(num|cat)__", "", str(x)) for x in pre.get_feature_names_out()]
def to_orig(n):
    if n.startswith("missingindicator_"):
        b_ = n[len("missingindicator_"):]; return b_ if b_ in PRIMARY_FEATURES else n
    if n in PRIMARY_FEATURES: return n
    c = [o for o in PRIMARY_FEATURES if n.startswith(o + "_")]
    return max(c, key=len) if c else n
def pretty(n):
    if n.startswith("missingindicator_"): return "Missing: " + LABELS.get(to_orig(n), n)
    o = to_orig(n); rest = n[len(o) + 1:] if n != o else ""
    return LABELS.get(o, o) + (f": {rest}" if rest else "")
orig_of = np.array([to_orig(n) for n in enc_names])

rng = np.random.default_rng(SEED)
idx_s = rng.choice(len(X_test), size=min(N_SHAP, len(X_test)), replace=False)
Xs = pre.transform(X_test.iloc[idx_s])
if BEST == "Logistic Regression":
    expl = shap.LinearExplainer(mdl, pre.transform(X_train.sample(min(1000, len(X_train)), random_state=SEED))); sv = expl.shap_values(Xs)
else:
    expl = shap.TreeExplainer(mdl); sv = expl.shap_values(Xs, check_additivity=False)
if isinstance(sv, list): sv = sv[-1]
sv = np.asarray(sv)
if sv.ndim == 3: sv = sv[:, :, -1]
base_v = float(np.ravel(expl.expected_value)[-1])

# additivity check on the model raw margin (probability scale for random forest)
if BEST == "Logistic Regression": margin = mdl.decision_function(Xs)
elif BEST == "CatBoost": margin = mdl.predict(Xs, prediction_type="RawFormulaVal")
elif BEST == "XGBoost": margin = mdl.get_booster().predict(xgb.DMatrix(Xs), output_margin=True)
elif BEST == "LightGBM": margin = mdl.predict(Xs, raw_score=True)
else: margin = mdl.predict_proba(Xs)[:, 1]
gap_ = margin - (base_v + sv.sum(1))
add_rep = pd.DataFrame({"Metric": ["SHAP sample size", "Mean abs gap", "Max abs gap", "Relative gap"],
                        "Value": [len(idx_s), np.abs(gap_).mean(), np.abs(gap_).max(), np.abs(gap_).mean() / (np.abs(margin).mean() + 1e-9)]})
save_table(add_rep, "shap_additivity_check"); print(add_rep.to_string(index=False))
R.update({"shap_sample_n": len(idx_s), "shap_additivity_max_gap": float(np.abs(gap_).max())})

sv_d = -sv                                   # model predicts survival; flip so positive = pushes toward death
# aggregated: sum signed SHAP over all columns of one variable per patient, THEN mean |.|
agg = pd.DataFrame({o: sv_d[:, orig_of == o].sum(1) for o in pd.unique(orig_of)})
imp = agg.abs().mean().sort_values(ascending=False)
S4 = pd.DataFrame({"Rank": range(1, len(imp) + 1), "Clinical feature": [LABELS.get(o, o) for o in imp.index], "Original feature": imp.index,
                   "Aggregated mean |SHAP|": imp.values, "Share of total (%)": 100 * imp.values / imp.values.sum()})
save_table(S4, "TableS4_shap_aggregated_all_predictors"); print(S4.head(12).round(4).to_string(index=False))
R["shap_top10"] = S4.head(10)[["Clinical feature", "Aggregated mean |SHAP|"]].round(4).to_dict("records")

top = S4.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 7)); yp = np.arange(len(top))
ax.hlines(yp, 0, top["Aggregated mean |SHAP|"], color="#999", lw=1.4)
ax.scatter(top["Aggregated mean |SHAP|"], yp, s=130, c=[DCOL.get(DOMAIN.get(o, ""), "#888") for o in top["Original feature"]], edgecolor="k", lw=0.7, zorder=3)
ax.set_yticks(yp); ax.set_yticklabels(top["Clinical feature"]); ax.set_xlabel("Aggregated mean |SHAP| (log-odds of death within 36 months)")
ax.grid(axis="x", ls=":", alpha=0.4); [ax.spines[s].set_visible(False) for s in ["top", "right", "left"]]
doms = sorted({DOMAIN.get(o, "Other") for o in top["Original feature"]})
ax.legend(handles=[plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=DCOL.get(d_, "#888"), markeredgecolor="k", markersize=9, label=d_) for d_ in doms],
          title="Domain", loc="lower right", fontsize=9)
save_fig(fig, "Fig_shap_aggregated")

# SF2: dummy-level beeswarm. shows top 14 encoded columns + "sum of other features"; values = log-odds of death
expl_obj = shap.Explanation(values=sv_d, base_values=np.full(len(sv_d), -base_v), data=Xs, feature_names=[pretty(n) for n in enc_names])
plt.figure(figsize=(10, 7)); shap.plots.beeswarm(expl_obj, max_display=14, show=False)
save_fig(plt.gcf(), "SF2_shap_beeswarm")

# direction of nodal variables, from the SHAP values (mean SHAP toward death)
dirs = []
for f in ["regional_nodes_positive", "regional_nodes_examined", "node_ratio"]:
    j = [k for k, n in enumerate(enc_names) if n == f]
    if j:
        v = Xs[:, j[0]]; lo_, hi_ = np.quantile(v, [1/3, 2/3])
        dirs.append({"Feature": f, "Mean SHAP(death), low third": sv_d[v <= lo_, j[0]].mean(), "Mean SHAP(death), high third": sv_d[v >= hi_, j[0]].mean()})
for k, n in enumerate(enc_names):
    if n.startswith("nodal_assessment_"):
        on = Xs[:, k] > 0.5
        if on.sum(): dirs.append({"Feature": n, "Mean SHAP(death), when present": sv_d[on, k].mean(), "N in sample": int(on.sum())})
nodal_dir = pd.DataFrame(dirs); save_table(nodal_dir, "TableS4b_nodal_shap_direction"); print(nodal_dir.round(4).to_string(index=False))

In [ ]:
# SECONDARY robustness checks only. The primary endpoint is the fixed-horizon 36-month CSS classification above.
ff = feat_full.copy()
tr_f, te_f = ff["year_numeric"].isin(TRAIN_YEARS), ff["year_numeric"].eq(TEST_YEAR)

# --- Cox PH (cause-specific), not interpreted as hazard ratios
cov = ["age_numeric", "sex_clean", "race_clean", "grade_clean", "stage_group_clean"]
def cox_design(df, cols=None):
    d_ = df[cov].copy(); d_["age_numeric"] = d_["age_numeric"].fillna(d_["age_numeric"].median())
    x = pd.concat([d_[["age_numeric"]], pd.get_dummies(d_[[c for c in cov if c != "age_numeric"]], drop_first=True).astype(float)], axis=1)
    return x if cols is None else x.reindex(columns=cols, fill_value=0.0)
xtr = cox_design(ff.loc[tr_f]); xte = cox_design(ff.loc[te_f], xtr.columns)
dtr = xtr.assign(duration=ff.loc[tr_f, "survival_months"].clip(lower=0.01).values, event=ff.loc[tr_f, "death_event_crc"].values)
dte = xte.assign(duration=ff.loc[te_f, "survival_months"].clip(lower=0.01).values, event=ff.loc[te_f, "death_event_crc"].values)
cph = CoxPHFitter(penalizer=0.1).fit(dtr, "duration", "event")
risk = cph.predict_partial_hazard(xte).values.ravel()
c_index = float(concordance_index(dte["duration"], -risk, dte["event"]))
s36 = cph.predict_survival_function(xte, times=[HORIZON]).T[HORIZON].values
defined = ff.loc[te_f, TARGET].notna().values
cox_auc = float(roc_auc_score(1 - ff.loc[te_f, TARGET].values[defined].astype(int), 1 - s36[defined]))
try:
    ph = proportional_hazard_test(cph, dtr, time_transform="rank").summary.reset_index()
    ph.to_csv(TAB / "cox_schoenfeld_test.csv", index=False); n_ph_viol = int((ph["p"] < 0.05).sum())
except Exception as e:
    print("Schoenfeld test skipped:", e); n_ph_viol = None
print(f"Cox C-index (2020, full censored cohort N={len(dte):,}) {c_index:.4f}; 3-yr AUC on analytic rows {cox_auc:.4f}; PH violations at p<0.05: {n_ph_viol}")

# --- Kaplan-Meier by stage (2020)
fig, ax = plt.subplots(figsize=(7, 6)); km = KaplanMeierFitter()
for s, g in ff.loc[te_f].groupby("stage_group_clean"):
    if len(g) >= MIN_SUBGROUP_N: km.fit(g["survival_months"].clip(lower=0.01), g["death_event_crc"], label=str(s)); km.plot_survival_function(ax=ax, ci_show=False)
ax.set_xlabel("Months since diagnosis"); ax.set_ylabel("CRC-specific survival probability"); save_fig(fig, "SF3_kaplan_meier_by_stage")

# --- Aalen-Johansen CIF by stage, 2018-2019, administratively censored at 48 months. fixed seed -> same jitter every run
CIF_H = 48
cif = ff.loc[tr_f].copy(); cif["code"] = 0
cif.loc[cif["death_event_crc"].eq(1), "code"] = 1
cif.loc[cif[VITAL].str.strip().eq("Dead") & cif["death_event_crc"].eq(0), "code"] = 2
cif.loc[cif["survival_months"] > CIF_H, "code"] = 0
cif["dur"] = cif["survival_months"].clip(lower=0.01, upper=CIF_H)
def cif_at(a, t):
    cd = a.cumulative_density_; ts = cd.index[cd.index <= t]
    return float(cd.loc[ts.max()].iloc[0]) if len(ts) else 0.0
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True); rows = []
for s, g in cif.groupby("stage_group_clean"):
    if len(g) < MIN_SUBGROUP_N: continue
    a1 = AalenJohansenFitter(calculate_variance=False, seed=SEED).fit(g["dur"], g["code"], event_of_interest=1, label=str(s)); a1.plot(ax=axes[0])
    a2 = AalenJohansenFitter(calculate_variance=False, seed=SEED).fit(g["dur"], g["code"], event_of_interest=2, label=str(s)); a2.plot(ax=axes[1])
    rows.append({"Stage group": str(s), "N": len(g), "CRC-death CIF @36mo": round(cif_at(a1, HORIZON), 4), "Other-cause death CIF @36mo": round(cif_at(a2, HORIZON), 4)})
axes[0].set_title("CRC-specific death"); axes[1].set_title("Other-cause death")
for a in axes: a.set_xlabel("Months since diagnosis"); a.set_ylabel("Cumulative incidence"); a.set_xlim(0, CIF_H)
save_fig(fig, "SF4_cif_by_stage")
S12 = pd.DataFrame(rows); save_table(S12, "TableS12_cif_36mo_by_stage"); print(S12.to_string(index=False))

# --- IPCW Brier and time-dependent AUC at 36 months, full 2020 cohort scored with the unchanged headline model
s36_full = score(BEST_PIPE, BEST_PLATT, ff.loc[te_f, PRIMARY_FEATURES])
dur_te = ff.loc[te_f, "survival_months"].clip(lower=0.01).values; ev_te = ff.loc[te_f, "death_event_crc"].values
dur_tr = ff.loc[tr_f, "survival_months"].clip(lower=0.01).values; ev_tr = ff.loc[tr_f, "death_event_crc"].values
kmc = KaplanMeierFitter().fit(dur_tr, 1 - ev_tr)                      # reverse KM for the censoring distribution
G = lambda t: np.clip(kmc.survival_function_at_times(np.atleast_1d(t)).values, 1e-3, 1.0)
a_ = (dur_te <= HORIZON) & (ev_te == 1); b_ = dur_te > HORIZON
contrib = np.zeros(len(dur_te)); contrib[a_] = (s36_full[a_] ** 2) / G(dur_te[a_]); contrib[b_] = ((1 - s36_full[b_]) ** 2) / G(HORIZON)[0]
brier_manual = float(contrib.sum() / len(dur_te))
brier_sk = td_auc = np.nan
if HAS_SKSURV:
    ytr_s, yte_s = Surv.from_arrays(ev_tr.astype(bool), dur_tr), Surv.from_arrays(ev_te.astype(bool), dur_te)
    try: brier_sk = float(sk_brier(ytr_s, yte_s, s36_full.reshape(-1, 1), np.array([float(HORIZON)]))[1][0])
    except Exception as e: print("sksurv brier failed:", e)
    try: td_auc = float(sk_cdauc(ytr_s, yte_s, 1 - s36_full, np.array([float(HORIZON)]))[0][0])
    except Exception as e: print("sksurv time-dependent AUC failed:", e)
print(f"IPCW Brier@36: sksurv {brier_sk:.4f}, manual {brier_manual:.4f}; time-dependent AUC@36 {td_auc:.4f}")

# --- censoring bounds: how far can the headline AUC move if censored patients were all survivors / all non-survivors
pd_full = 1 - s36_full
auc_best = float(roc_auc_score(1 - ff.loc[te_f, TARGET_BEST].values, pd_full)); auc_worst = float(roc_auc_score(1 - ff.loc[te_f, TARGET_WORST].values, pd_full))
lo_b, hi_b = sorted([auc_best, auc_worst]); head_auc = R["headline_auc_roc"]
inside = lo_b <= head_auc <= hi_b
print(f"Censoring bounds AUC [{lo_b:.4f}, {hi_b:.4f}]; headline {head_auc:.4f} lies {'INSIDE' if inside else 'OUTSIDE'} the bounds")

S11 = pd.DataFrame([
    {"Analysis (SECONDARY)": "Cox PH concordance, 2020 full censored cohort", "Value": c_index, "N": len(dte)},
    {"Analysis (SECONDARY)": "Cox PH 3-year AUC on analytic test rows", "Value": cox_auc, "N": int(defined.sum())},
    {"Analysis (SECONDARY)": "IPCW Brier @36 mo (scikit-survival)", "Value": brier_sk, "N": len(dte)},
    {"Analysis (SECONDARY)": "IPCW Brier @36 mo (manual reverse-KM)", "Value": brier_manual, "N": len(dte)},
    {"Analysis (SECONDARY)": "Time-dependent AUC @36 mo", "Value": td_auc, "N": len(dte)},
    {"Analysis (SECONDARY)": "Headline AUC (analytic test set)", "Value": head_auc, "N": len(y_test)},
    {"Analysis (SECONDARY)": "Best-case censoring bound AUC", "Value": auc_best, "N": int(te_f.sum())},
    {"Analysis (SECONDARY)": "Worst-case censoring bound AUC", "Value": auc_worst, "N": int(te_f.sum())},
    {"Analysis (SECONDARY)": "Headline inside best/worst bounds", "Value": float(inside), "N": np.nan}])
save_table(S11, "TableS11_secondary_time_to_event_and_censoring")
R.update({"cox_c_index": c_index, "cox_auc_3yr": cox_auc, "cox_ph_violations": n_ph_viol, "ipcw_brier_sksurv": brier_sk, "ipcw_brier_manual": brier_manual,
          "time_dependent_auc_36": td_auc, "censoring_bound_best": auc_best, "censoring_bound_worst": auc_worst, "headline_inside_bounds": bool(inside),
          "cif_36mo": S12.set_index("Stage group")["CRC-death CIF @36mo"].to_dict()})

In [ ]:
# 2018 -> 2019, same model family and tuned parameters
m18, m19 = (feat["year_numeric"] == 2018).to_numpy(), (feat["year_numeric"] == 2019).to_numpy()
X18, y18 = feat.loc[m18, PRIMARY_FEATURES], feat.loc[m18, TARGET].to_numpy(int)
X19, y19 = feat.loc[m19, PRIMARY_FEATURES], feat.loc[m19, TARGET].to_numpy(int)
pipe18, platt18, _ = fit_platt(X18, y18, BEST, BEST_PARAMS[BEST])
pd19 = 1 - score(pipe18, platt18, X19)
excl = S2.set_index("Year of diagnosis")["Excluded %"]
S14 = pd.DataFrame([
    {"Split": "2018 train -> 2019 test (pre-pandemic)", "Train N": len(X18), "Test N": len(X19), "Train excluded %": excl.get(2018), "Test excluded %": excl.get(2019), **{k: v for k, v in evaluate(1 - y19, pd19).items() if k in ["AUC-ROC", "AUC-PR", "Brier", "F1"]}},
    {"Split": "2018-2019 train -> 2020 test (headline)", "Train N": len(X_train), "Test N": len(X_test), "Train excluded %": round((excl.get(2018) + excl.get(2019)) / 2, 2), "Test excluded %": excl.get(2020), **{k: v for k, v in evaluate(yd_test, pd_head).items() if k in ["AUC-ROC", "AUC-PR", "Brier", "F1"]}}])
save_table(S14, "TableS14_prepandemic_sensitivity"); print(S14.round(4).to_string(index=False))
R.update({"auc_2019_prepandemic": float(S14.loc[0, "AUC-ROC"]), "auc_gap_2020_minus_2019": float(S14.loc[1, "AUC-ROC"] - S14.loc[0, "AUC-ROC"])})

# nodal missingness: complete case = nodal_assessment "Examined, count known"
cc_tr = (X_train["nodal_assessment"] == "Examined, count known").to_numpy(); cc_te = (X_test["nodal_assessment"] == "Examined, count known").to_numpy()
pipe_cc, platt_cc, _ = fit_platt(X_train[cc_tr], y_train[cc_tr], BEST, BEST_PARAMS[BEST])
pd_cc = 1 - score(pipe_cc, platt_cc, X_test[cc_te])
scen = [("(a) Full model, all test rows (headline)", yd_test, pd_head), ("(b) Full model, complete-case rows", yd_test[cc_te], pd_head[cc_te]),
        ("(c) Complete-case-trained model, complete-case rows", yd_test[cc_te], pd_cc), ("(d) Full model, other rows (nodal info missing/special)", yd_test[~cc_te], pd_head[~cc_te])]
S13 = pd.DataFrame([{"Scenario": l, "N": len(y_), "Death rate": float(np.mean(y_)), **{k: v for k, v in evaluate(y_, p_).items() if k in ["AUC-ROC", "AUC-PR", "Brier", "Precision", "Recall", "F1"]}} for l, y_, p_ in scen])
dcc = bootstrap(yd_test[cc_te], lambda i: roc_auc_score(yd_test[cc_te][i], pd_head[cc_te][i]) - roc_auc_score(yd_test[cc_te][i], pd_cc[i])); cm_, cl_, ch_ = ci(dcc)
S13["Delta AUC (b - c)"] = [""] * 4; S13.loc[1, "Delta AUC (b - c)"] = f"{cm_:+.4f} ({cl_:+.4f} to {ch_:+.4f})"
save_table(S13, "TableS13_nodal_missingness_sensitivity"); print(S13.round(4).to_string(index=False))
dr_cc, dr_non = float(yd_test[cc_te].mean()), float(yd_test[~cc_te].mean())
share_non = 100 * (1 - cc_te.mean())
mnar_text = (f"Complete nodal information (nodes examined with a known count) was available for {100*cc_te.mean():.1f}% of the 2020 test cohort; {share_non:.1f}% had no nodes examined, "
             f"an unknown nodal status or an unspecified count. The 36-month CRC death rate was {100*dr_cc:.1f}% in the complete-case group and {100*dr_non:.1f}% in the remainder, "
             f"so nodal documentation is related to outcome and the missingness is unlikely to be random. The complete-case comparison is a same-population check "
             f"(full vs complete-case-trained model on the same rows), not evidence that the complete-case subset represents the whole cohort.")
print(mnar_text)
R.update({"nodal_cc_share_test": float(cc_te.mean()), "death_rate_complete_case": dr_cc, "death_rate_non_complete_case": dr_non, "nodal_delta_auc_full_minus_cc": [cm_, cl_, ch_]})

In [ ]:
# determinism: refit the selected model and confirm the raw test AUC is identical
re_pipe = pipe_for(X_train, BEST, BEST_PARAMS[BEST]).fit(X_train, y_train)
auc_refit = roc_auc_score(yd_test, 1 - re_pipe.predict_proba(X_test)[:, 1]); auc_first = roc_auc_score(yd_test, 1 - P_RAW[BEST])
check("determinism: refit raw test AUC identical", abs(auc_refit - auc_first) < 1e-9, f"{auc_refit:.10f} vs {auc_first:.10f}")
check("headline AUC reproduces from saved objects", abs(roc_auc_score(yd_test, 1 - score(BEST_PIPE, BEST_PLATT, X_test)) - R["headline_auc_roc"]) < 1e-12)

STATEMENTS = {
 "Terminology": "The primary analysis is a fixed-horizon (36-month) CRC cancer-specific survival risk classification. Cox PH, Aalen-Johansen, IPCW and censoring-bound analyses are secondary companion checks.",
 "Class imbalance": f"Training negative:positive ratio {NEG_POS:.2f}:1 (CRC death : 3-year survivor). Class weighting was used for all five model families; no resampling.",
 "CV reuse": "Model selection and hyperparameter tuning used the same 5-fold CV on 2018-2019 data (no nested CV), so the training-CV AUC may be slightly optimistic. The 2020 test set was not used for selection, tuning, thresholds, calibrators or the conformal quantile.",
 "Patient independence": f"{len(ov_raw)} patients had records in both a training year and 2020 in the raw extract; after keeping first primaries {len(ov_first)} remained. Zero overlap between train and test patients was asserted.",
 "Pandemic": "2020 coincides with COVID-19 related disruption of screening and care, so the temporal test also reflects a pandemic-era distribution shift.",
 "Nodal missingness": mnar_text,
 "Pretreatment label": "The pretreatment-only model still contains variables derived from pathology or combined clinical/pathological staging (see pretreatment_set_caveats.csv); it approximates, not equals, a strictly pretreatment model.",
 "Censoring": f"Patients censored before 36 months were excluded from the primary analysis ({S2['Excluded %'].min():.1f}-{S2['Excluded %'].max():.1f}% by year, highest in 2020). Best/worst-case AUCs were {auc_best:.3f} and {auc_worst:.3f}; the headline lies {'inside' if inside else 'outside'} these bounds.",
 "Interpretation": "SHAP values describe how the fitted model uses the inputs; they are not causal effects."}
pd.DataFrame([{"Item": k, "Statement": v} for k, v in STATEMENTS.items()]).to_csv(TAB / "manuscript_ready_statements.csv", index=False)

# results_numbers.json + tidy csv
R["generated_at"] = datetime.now().isoformat(timespec="seconds"); R["versions"] = VERSIONS; R["quick_mode"] = QUICK
def clean(o):
    if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, (np.floating, float)): return None if np.isnan(o) else float(o)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    return o
Rc = clean(R)
(OUT / "results_numbers.json").write_text(json.dumps(Rc, indent=2))
flat = []
def walk(prefix, o):
    if isinstance(o, dict): [walk(f"{prefix}.{k}" if prefix else k, v) for k, v in o.items()]
    elif isinstance(o, list) and o and all(not isinstance(v, (dict, list)) for v in o): flat.append((prefix, json.dumps(o)))
    elif isinstance(o, list): [walk(f"{prefix}[{i}]", v) for i, v in enumerate(o)]
    else: flat.append((prefix, o))
walk("", Rc); pd.DataFrame(flat, columns=["key", "value"]).to_csv(OUT / "results_numbers.csv", index=False)

MAP = [("Headline AUC-ROC, AUC-PR, Brier, precision, recall, F1", "Abstract, Results (headline), Table 4"), ("Selected model / CV AUCs / paired AUC differences", "Results (model comparison), Table 3, S3"),
       ("TNM baselines and delta AUC", "Results, Table 5, Fig_dca"), ("Calibration slope/intercept/ECE", "Table 4, S5, Fig_calibration"), ("SHAP ranks (aggregated)", "Results, Fig_shap_aggregated, S4, SF2"),
       ("Stage-wise AUC and delta AUC", "S6, S7"), ("Subgroup AUCs and spread", "S8, Limitations"), ("Pretreatment AUC", "Results, S9"), ("Conformal coverage/abstain", "S10"),
       ("Cox / IPCW / bounds", "S11, Limitations"), ("CIF at 36 months", "S12"), ("Nodal sensitivity and death rates", "S13, Limitations"), ("Pre-pandemic AUC", "S14, Discussion"),
       ("Cohort counts, attrition, exclusion by year", "Methods flow, S1, S2"), ("Baseline characteristics (nodes, tumor size recoded)", "Table 2, S15"), ("Missingness percentages", "S16, Limitations")]
(OUT / "RESULTS_MAP.md").write_text("# Results map\n\nAll values are in results_numbers.json.\n\n| Result | Where it appears |\n|---|---|\n" + "\n".join(f"| {a} | {b} |" for a, b in MAP) + "\n")
vt = "# Verification\n\nQuick mode run: " + str(QUICK) + "\n\n| Check | Result | Detail |\n|---|---|---|\n" + "\n".join(f"| {n} | {'PASS' if ok else 'FAIL'} | {d_} |" for n, ok, d_ in CHECKS)
(OUT / "VERIFICATION.md").write_text(vt + f"\n\nFeature counts: primary {len(PRIMARY_FEATURES)}, pretreatment {len(PRETREATMENT_FEATURES)}.\n")
print(vt); print("\nOutputs written to", OUT.resolve())
if QUICK: print("WARNING: this was a QUICK_TEST_MODE run; do not report these numbers.")

In [ ]:
top = S4.head(15).iloc[::-1].reset_index(drop=True)      # largest at top
vals = top["Aggregated mean |SHAP|"].values
cols = plt.cm.cividis(np.linspace(0, 1, len(top)))[::-1]  # dark (large) -> yellow (small)

fig, ax = plt.subplots(figsize=(8.5, 6.5))
ax.barh(top["Clinical feature"], vals, color=cols, height=0.78)
for y, v in enumerate(vals):
    ax.text(v + 0.008, y, f"{v:.3f}", va="center", fontsize=8.5, color="#333")
ax.set_xlabel("Aggregated mean |SHAP value| (log-odds of death within 36 months)", fontsize=10)
ax.set_xlim(0, vals.max() * 1.12)
ax.grid(axis="x", ls=":", alpha=0.4); ax.set_axisbelow(True)
for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
ax.tick_params(axis="y", labelsize=10)
fig.tight_layout()
save_fig(fig, "Fig_shap_aggregated")